# Sample 200 iNaturalist bird images

This notebook filters the combined CUB/iNaturalist bridge manifest to `source_dataset == "inaturalist"`, selects a reproducible random sample of 200 rows, and copies the images through `copy_confirmed.py` while preserving their iNaturalist category directories. It then joins the sampled rows to the machine-generated bird bounding boxes by stable `bridge_image_id`.

Outputs are written to `Notebooks/iNaturalist Samples`:

- 200 images beneath their original category directories
- `sampled_manifest.csv`, containing the complete machine bounding-box manifest rows in sample order, with paths localized to the copied images
- `sampled_manifest_ground_truth.csv`, when present, is preserved and validated against the sampled IDs

Previously copied image directories are recreated on every run so they cannot retain stale images. Existing CSV and lock files in the output directory are preserved; source images and human ground truth are never modified.

## 1. Configuration and project paths

In [ ]:
from __future__ import annotations

import csv
import os
import random
import shutil
import sys
import tempfile
from pathlib import Path

SAMPLE_SIZE = 200
RANDOM_SEED = 20261014

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "copy_confirmed.py").is_file() and (candidate / "Notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root containing copy_confirmed.py")

PROJECT_ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = PROJECT_ROOT / "Notebooks" / "Outputs" / "Bridge Outputs" / "bird_bridge_manifest.csv"
DESTINATION = PROJECT_ROOT / "Notebooks" / "iNaturalist Samples"
SAMPLED_MANIFEST_PATH = DESTINATION / "sampled_manifest.csv"
GROUND_TRUTH_PATH = DESTINATION / "sampled_manifest_ground_truth.csv"
BBOX_MANIFEST_PATH = (
    PROJECT_ROOT
    / "Notebooks"
    / "Outputs"
    / "iNaturalist Outputs"
    / "Bounding Box Outputs"
    / "inaturalist_bird_bbox_manifest.csv"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bbox_data import load_dataset, load_ground_truth
from copy_confirmed import IMAGE_EXTENSIONS, copy_selected_files

print(f"Project root:  {PROJECT_ROOT}")
print(f"Manifest:      {MANIFEST_PATH}")
print(f"Machine boxes: {BBOX_MANIFEST_PATH}")
print(f"Destination:   {DESTINATION}")
print(f"Sample size:   {SAMPLE_SIZE}")
print(f"Random seed:   {RANDOM_SEED}")

## 2. Load, filter, validate, and sample the manifest

In [ ]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Bridge manifest not found: {MANIFEST_PATH}")

with MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    fieldnames = reader.fieldnames
    if fieldnames is None:
        raise ValueError("Bridge manifest has no CSV header")
    required_columns = {"source_dataset", "photos_directory", "photo_relative_path", "absolute_path"}
    missing_columns = required_columns.difference(fieldnames)
    if missing_columns:
        raise ValueError(f"Bridge manifest is missing columns: {sorted(missing_columns)}")
    inaturalist_rows = [
        row for row in reader
        if row["source_dataset"].strip().casefold() == "inaturalist"
    ]

if len(inaturalist_rows) < SAMPLE_SIZE:
    raise ValueError(f"Only {len(inaturalist_rows)} iNaturalist rows are available")

photo_roots = {Path(row["photos_directory"]).resolve() for row in inaturalist_rows}
if len(photo_roots) != 1:
    raise ValueError(f"Expected one iNaturalist photos directory, found: {sorted(map(str, photo_roots))}")
SOURCE_ROOT = photo_roots.pop()
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"iNaturalist photos directory not found: {SOURCE_ROOT}")

sampled_rows = random.Random(RANDOM_SEED).sample(inaturalist_rows, SAMPLE_SIZE)
relative_paths = [Path(row["photo_relative_path"]) for row in sampled_rows]
if len({str(path).casefold() for path in relative_paths}) != SAMPLE_SIZE:
    raise ValueError("The sampled manifest rows do not resolve to 200 unique relative paths")

path_mismatches = [
    row["bridge_image_id"]
    for row, relative_path in zip(sampled_rows, relative_paths)
    if (SOURCE_ROOT / relative_path).resolve() != Path(row["absolute_path"]).resolve()
]
missing_sources = [SOURCE_ROOT / path for path in relative_paths if not (SOURCE_ROOT / path).is_file()]
if path_mismatches:
    raise ValueError(f"Manifest path columns disagree for {len(path_mismatches)} sampled rows")
if missing_sources:
    raise FileNotFoundError(f"{len(missing_sources)} sampled source images are missing; first: {missing_sources[0]}")

print(f"iNaturalist candidates: {len(inaturalist_rows):,}")
print(f"Source directory:       {SOURCE_ROOT}")
print(f"Sampled rows:           {len(sampled_rows)}")
print(f"Sampled categories:     {len({path.parent for path in relative_paths})}")

## 3. Copy images and write the sampled bounding-box manifest

`copy_selected_files` uses Windows `robocopy`, preserves every relative category directory, and SHA-256 verifies each destination image against its source. The full machine manifest is streamed once and joined by `bridge_image_id`, avoiding a large in-memory read. Human ground truth and its lock file are preserved.

In [ ]:
DESTINATION.mkdir(parents=True, exist_ok=True)
for path in DESTINATION.iterdir():
    if path.is_dir():
        shutil.rmtree(path)
    elif path.suffix.lower() in IMAGE_EXTENSIONS:
        path.unlink()

copy_result = copy_selected_files(
    SOURCE_ROOT,
    DESTINATION,
    relative_paths,
    verify=True,
)

if not BBOX_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Machine bounding-box manifest not found: {BBOX_MANIFEST_PATH}")

sample_order_by_id = {
    row["bridge_image_id"]: sample_order
    for sample_order, row in enumerate(sampled_rows, start=1)
}
if len(sample_order_by_id) != SAMPLE_SIZE:
    raise ValueError("Sampled rows do not have unique bridge_image_id values")

bbox_rows_by_id = {}
with BBOX_MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    bbox_fieldnames = reader.fieldnames
    if bbox_fieldnames is None:
        raise ValueError("Machine bounding-box manifest has no CSV header")
    required_bbox_columns = {
        "bridge_image_id", "photo_relative_path", "bbox_x", "bbox_y",
        "bbox_width", "bbox_height", "bbox_detection_status",
    }
    missing_bbox_columns = required_bbox_columns.difference(bbox_fieldnames)
    if missing_bbox_columns:
        raise ValueError(
            f"Machine bounding-box manifest is missing columns: {sorted(missing_bbox_columns)}"
        )
    for row in reader:
        bridge_image_id = row["bridge_image_id"]
        if bridge_image_id not in sample_order_by_id:
            continue
        if bridge_image_id in bbox_rows_by_id:
            raise ValueError(f"Duplicate machine bounding-box row: {bridge_image_id}")
        bbox_rows_by_id[bridge_image_id] = row

missing_bbox_ids = sample_order_by_id.keys() - bbox_rows_by_id.keys()
if missing_bbox_ids:
    raise ValueError(
        f"Machine bounding-box manifest is missing {len(missing_bbox_ids)} sampled IDs; "
        f"first: {sorted(missing_bbox_ids)[0]}"
    )

output_fieldnames = ["sample_order", *bbox_fieldnames]
temporary_manifest = None
try:
    with tempfile.NamedTemporaryFile(
        "w", encoding="utf-8-sig", newline="",
        prefix=f".{SAMPLED_MANIFEST_PATH.name}.", suffix=".tmp",
        dir=DESTINATION, delete=False,
    ) as handle:
        temporary_manifest = Path(handle.name)
        writer = csv.DictWriter(handle, fieldnames=output_fieldnames, lineterminator="\n")
        writer.writeheader()
        for sample_order, sampled_row in enumerate(sampled_rows, start=1):
            bridge_image_id = sampled_row["bridge_image_id"]
            row = bbox_rows_by_id[bridge_image_id].copy()
            relative_path = Path(row["photo_relative_path"])
            if relative_path != Path(sampled_row["photo_relative_path"]):
                raise ValueError(f"Sample and machine paths disagree for {bridge_image_id}")
            row["photos_directory"] = str(DESTINATION.resolve())
            row["image_directory"] = str((DESTINATION / relative_path.parent).resolve())
            row["absolute_path"] = str((DESTINATION / relative_path).resolve())
            writer.writerow({"sample_order": sample_order, **row})
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary_manifest, SAMPLED_MANIFEST_PATH)
    temporary_manifest = None
finally:
    if temporary_manifest is not None:
        temporary_manifest.unlink(missing_ok=True)

print(copy_result)
print(f"Sample manifest with machine boxes: {SAMPLED_MANIFEST_PATH}")
print(f"Preserved ground truth:            {GROUND_TRUTH_PATH.exists()}")

## 4. Validate the completed sample, machine boxes, and ground truth

In [ ]:
copied_images = sorted(
    path for path in DESTINATION.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
with SAMPLED_MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    written_rows = list(csv.DictReader(handle))

sampled_dataset = load_dataset(SAMPLED_MANIFEST_PATH)
written_ids = [row["bridge_image_id"] for row in written_rows]
ground_truth = load_ground_truth(GROUND_TRUTH_PATH)

assert copy_result.files_queued == SAMPLE_SIZE
assert copy_result.verified_files == SAMPLE_SIZE
assert len(copied_images) == SAMPLE_SIZE
assert len(written_rows) == SAMPLE_SIZE
assert len(sampled_dataset) == SAMPLE_SIZE
assert len(set(written_ids)) == SAMPLE_SIZE
assert written_ids == [row["bridge_image_id"] for row in sampled_rows]
assert {row["source_dataset"] for row in written_rows} == {"inaturalist"}
assert all((DESTINATION / path).is_file() for path in relative_paths)
assert all(sampled_dataset.resolve_image_path(record).is_file() for record in sampled_dataset)
assert set(ground_truth).issubset(written_ids)
assert all(
    ground_truth[source_id].relative_path == row["photo_relative_path"]
    for source_id, row in ((row["bridge_image_id"], row) for row in written_rows)
    if source_id in ground_truth
)

print(f"Validated {len(copied_images)} copied and hash-verified iNaturalist images.")
print(f"Machine boxes: {sum(record.has_bbox for record in sampled_dataset)} detected; "
      f"{sum(not record.has_bbox for record in sampled_dataset)} without a retained box.")
print(f"Ground truth:  {len(ground_truth)} compatible review records.")
print(f"Preserved {len({path.parent for path in copied_images})} category directories.")
print(f"Output: {DESTINATION}")